In [ ]:
from pathlib import Path
import pandas as pd
from collections import defaultdict
import mmh3
import numpy as np
from pypdf import PdfReader

# Define your directory path
doc_dir = Path("./Document")

# Find all PDF files in the folder
pdf_files = list(doc_dir.glob("*.pdf"))

raw_documents = []

for file_path in pdf_files:
    print(f"Reading PDF: {file_path.name}")
    try:
        reader = PdfReader(file_path)
        doc_text = ""
        
        # Loop through every page and extract text
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                doc_text += page_text + "\n"
        
        if doc_text.strip():
            raw_documents.append(doc_text)
            print(f"  -> Successfully extracted {len(doc_text)} characters.")
        else:
            print(f"  -> Warning: No text found (might be scanned images).")
            
    except Exception as e:
        print(f"  -> Error reading {file_path.name}: {e}")

print(f"\nSuccessfully loaded {len(raw_documents)} valid text documents into memory!")
print(raw_documents)

import re
import string


def preprocess_text(text: str) -> str:
    """Preprocesses raw document text:

    1. Removes EOL characters (newlines/carriage returns) by replacing them with spaces.
    2. Converts text to lowercase.
    3. Removes all punctuation.
    4. Normalizes extra whitespace.
    """
    if not isinstance(text, str):
        return ""

    # 1. Remove EOL characters (replace with space to keep word boundaries intact)
    text = text.replace("\r\n", " ").replace("\n", " ").replace("\r", " ")

    # 2. Convert to lowercase
    text = text.lower()

    # 3. Remove punctuation using translation tables (very fast)
    text = text.translate(str.maketrans("", "", string.punctuation))

    # 4. Clean up any extra multi-spaces left behind
    text = re.sub(r"\s+", " ", text).strip()

    return text



# Clean every document in your list
cleaned_documents = [preprocess_text(doc) for doc in raw_documents]

for idx, doc in enumerate(cleaned_documents):
    print(f"Doc {idx} Cleaned: {doc}")




def text_to_hashed_shingles(
    text: str, k: int = 9, shingle_seed: int = 42
):

    shingles = {text[i : i + k] for i in range(len(text) - k + 1)}

    hashed_list = [
        mmh3.hash(shingle, seed=shingle_seed, signed=False) for shingle in shingles
    ]

    # print(shingles)
    # print(hashed_list)

    # Convert directly to a fast 1D array
    return np.array(hashed_list, dtype=np.uint32)


def generate_signature_matrix(
    documents_shingles: list[np.ndarray], num_hashes: int = 100
):
    
    num_docs = len(documents_shingles)
    signature_matrix = np.empty((num_hashes, num_docs), dtype=np.uint32)

    for hash_idx in range(num_hashes):
        for doc_idx, shingle_array in enumerate(documents_shingles):
            # if len(shingle_array) == 0:
            #     signature_matrix[hash_idx, doc_idx] = np.iinfo(np.uint32).max
            #     continue

            # Pass the byte representation of each hashed uint32 into mmh3 with a unique seed
            scrambled_hashes = [
                mmh3.hash(x.tobytes(), seed=hash_idx, signed=False)
                for x in shingle_array
            ]
            signature_matrix[hash_idx, doc_idx] = np.min(scrambled_hashes)

    return signature_matrix
